# Model Data Preparation and Feature Selection

This notebook prepares a model-ready dataset from the processed water-quality master file and stops after feature selection and handoff.

## 1. Load water_quality_master.csv

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
ROOT = Path.cwd()
if not (ROOT / 'data').exists():
    ROOT = ROOT.parent
PROCESSED_DIR = ROOT / 'data' / 'processed'
water_master = pd.read_csv(PROCESSED_DIR / 'water_quality_master.csv')
water_master.head()

,state,station_name,year,dissolved_oxygen,bod,fecal_coliform,source_file
0,Uttarakhand,GANGA AT HARIDWAR D/S,2011.0,6.7,5.6,1150.0,rs_session-241_as196_1.1.csv
1,Uttarakhand,GANGA AT HARIDWAR D/S,2012.0,7.2,5.3,NaN,rs_session-241_as196_1.1.csv
2,Uttarakhand,GANGA AT HARIDWAR D/S,2013.0,6.5,5.2,NaN,rs_session-241_as196_1.1.csv
3,Uttarakhand,GANGA AT HARIDWAR D/S,2014.0,5.0,5.2,NaN,rs_session-241_as196_1.1.csv
4,Uttarakhand,GANGA AT HARIDWAR D/S,2015.0,9.2,2.8,580.0,rs_session-241_as196_1.1.csv


## 2. Inspect target availability

In [2]:
print(water_master[['dissolved_oxygen','bod','fecal_coliform']].notna().sum())
print(water_master['year'].isna().sum())

dissolved_oxygen    410
bod                  50
fecal_coliform      382
dtype: int64
84


## 3. Identify usable records

In [3]:
usable = water_master[water_master['year'].notna()].copy()
print(usable.shape)
print(usable[['dissolved_oxygen','bod','fecal_coliform']].notna().sum())

(329, 7)
dissolved_oxygen    326
bod                  50
fecal_coliform      302
dtype: int64


## 4. Decide/document target variables

In [4]:
target_notes = pd.DataFrame([
    {'target_variable':'dissolved_oxygen','usable':True,'reason':'Available across all source files, including the historical wide file.'},
    {'target_variable':'fecal_coliform','usable':True,'reason':'Available across all source files, though some rows are missing.'},
    {'target_variable':'bod','usable':False,'reason':'Missing entirely from the 2018-2020 files and unavailable in the yearless file.'},
])
display(target_notes)

,target_variable,usable,reason
0,dissolved_oxygen,True,"Available across all source files, including t..."
1,fecal_coliform,True,"Available across all source files, though some..."
2,bod,False,Missing entirely from the 2018-2020 files and ...


## 5. Filter unusable records according to explicit rules

In [5]:
model_dataset = usable[usable[['dissolved_oxygen','fecal_coliform']].notna().any(axis=1)].copy()
print(model_dataset.shape)

(326, 7)


## 6. Inspect distributions

In [6]:
display(model_dataset[['year','dissolved_oxygen','bod','fecal_coliform']].describe(include='all'))

,year,dissolved_oxygen,bod,fecal_coliform
count,326.000000,326.000000,50.000000,302.000000
mean,2018.088957,7.889387,4.470000,24577.937748
std,2.361499,1.076114,1.373399,62513.107982
min,2011.000000,5.000000,2.800000,1.800000
25%,2018.000000,7.300000,3.625000,1700.000000
50%,2019.000000,7.900000,4.150000,6100.000000
75%,2020.000000,8.500000,4.975000,19875.000000
max,2020.000000,10.600000,8.400000,592500.000000


## 7. Inspect correlations where appropriate

In [7]:
numeric_cols = ['year','dissolved_oxygen','bod','fecal_coliform']
display(model_dataset[numeric_cols].corr(numeric_only=True))

,year,dissolved_oxygen,bod,fecal_coliform
year,1.000000,0.139735,-0.329735,-0.179927
dissolved_oxygen,0.139735,1.000000,-0.466569,-0.420422
bod,-0.329735,-0.466569,1.000000,0.019843
fecal_coliform,-0.179927,-0.420422,0.019843,1.000000


## 8. Identify candidate numerical features

In [8]:
candidate_features = ['year', 'dissolved_oxygen', 'bod']
print(candidate_features)

['year', 'dissolved_oxygen', 'bod']


## 9. Remove identifiers / metadata columns that should not be ML features

In [9]:
feature_frame = model_dataset[['year','dissolved_oxygen','bod','fecal_coliform']].copy()
print(feature_frame.columns.tolist())

['year', 'dissolved_oxygen', 'bod', 'fecal_coliform']


## 10. Perform feature selection

In [10]:
selected_features = ['year', 'dissolved_oxygen', 'bod']
print(selected_features)

['year', 'dissolved_oxygen', 'bod']


## 11. Produce a final model-ready dataset

In [11]:
model_ready = model_dataset[['state','year','dissolved_oxygen','bod','fecal_coliform','source_file']].copy()
print(model_ready.shape)
display(model_ready.head())

(326, 6)


,state,year,dissolved_oxygen,bod,fecal_coliform,source_file
0,Uttarakhand,2011.0,6.7,5.6,1150.0,rs_session-241_as196_1.1.csv
1,Uttarakhand,2012.0,7.2,5.3,NaN,rs_session-241_as196_1.1.csv
2,Uttarakhand,2013.0,6.5,5.2,NaN,rs_session-241_as196_1.1.csv
3,Uttarakhand,2014.0,5.0,5.2,NaN,rs_session-241_as196_1.1.csv
4,Uttarakhand,2015.0,9.2,2.8,580.0,rs_session-241_as196_1.1.csv


## 12. Export the feature list and model-ready dataset

In [12]:
feature_docs = pd.DataFrame([
    {'feature_name':'state','feature_type':'categorical','meaning':'State where the monitoring station is located','source':'water_quality_master.csv','reason_for_inclusion':'Geographic signal that may explain regional variation'},
    {'feature_name':'year','feature_type':'numeric/ordinal','meaning':'Observation year','source':'water_quality_master.csv','reason_for_inclusion':'Captures the time dimension of the historical records'},
    {'feature_name':'dissolved_oxygen','feature_type':'numeric','meaning':'Observed dissolved oxygen concentration','source':'water_quality_master.csv','reason_for_inclusion':'Core water-quality variable and a candidate model input or target'},
    {'feature_name':'bod','feature_type':'numeric','meaning':'Biochemical oxygen demand','source':'water_quality_master.csv','reason_for_inclusion':'Potential explanatory variable where available'},
    {'feature_name':'fecal_coliform','feature_type':'numeric','meaning':'Observed fecal coliform concentration','source':'water_quality_master.csv','reason_for_inclusion':'Potential target or response variable for future modeling'},
])
feature_docs

,feature_name,feature_type,meaning,source,reason_for_inclusion
0,state,categorical,State where the monitoring station is located,water_quality_master.csv,Geographic signal that may explain regional va...
1,year,numeric/ordinal,Observation year,water_quality_master.csv,Captures the time dimension of the historical ...
2,dissolved_oxygen,numeric,Observed dissolved oxygen concentration,water_quality_master.csv,Core water-quality variable and a candidate mo...
3,bod,numeric,Biochemical oxygen demand,water_quality_master.csv,Potential explanatory variable where available
4,fecal_coliform,numeric,Observed fecal coliform concentration,water_quality_master.csv,Potential target or response variable for futu...


### Handoff note
This notebook ends at feature selection. No model training or evaluation is performed here.